In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 🧠 Face Mask Detection — Full CNN Pipeline (v2)
### Deep Learning with Keras + OpenCV | Production-Optimised Notebook

---

## 📋 Table of Contents
1. [Environment Setup & Imports](#1-environment-setup)
2. [Speed Setup: Copy Data + Mixed Precision](#2-speed-setup)
3. [Dataset Audit (train / val / test)](#3-dataset-audit)
4. [Preprocessing & tf.data Pipeline](#4-pipeline)
5. [CNN Architecture — Scratch with Residual Blocks](#5-cnn-architecture)
6. [KerasTuner Hyperband — HPO](#6-hyperband)
7. [Final Training with Best Hyperparameters](#7-training)
8. [Training Curves & Metrics](#8-metrics)
9. [Validation Evaluation](#9-val-evaluation)
10. [Test Set Evaluation — Unbiased Final Metrics](#10-test-evaluation)
11. [MaxPool Feature Map Visualisation](#11-feature-maps)
12. [Grad-CAM Explainability](#12-gradcam)
13. [Save Model & config.json](#13-save)
14. [Real-Time Detection — Webcam / Video](#14-realtime)

---

### 🗂️ Dataset Folder Structure (pre-split)
```
data/
├── train/
│   ├── with_mask/
│   └── without_mask/
├── val/
│   ├── with_mask/
│   └── without_mask/
└── test/
    ├── with_mask/
    └── without_mask/
```
> **Dataset:** [Kaggle — Face Mask Detection](https://www.kaggle.com/datasets/omkargurav/face-mask-dataset)


## 1. Environment Setup & Imports <a id='1-environment-setup'></a>

In [ ]:
# ─── Install extra dependencies (run once) ──────────────────────────────────
!pip install keras-tuner opencv-python-headless --quiet

In [ ]:
import os, json, time, random, warnings
warnings.filterwarnings('ignore')

import numpy as np
import cv2
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import keras_tuner as kt

from sklearn.metrics import (
    classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_curve, auc,
    accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.utils import class_weight as sk_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print(f"TensorFlow  : {tf.__version__}")
print(f"Keras       : {keras.__version__}")
print(f"OpenCV      : {cv2.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"GPU devices : {tf.config.list_physical_devices('GPU')}")

---
## 2. Speed Setup: Copy Data + Mixed Precision <a id='2-speed-setup'></a>

**Root cause of 15 s/step:** every image was read over the Google Drive network mount.
Copying to Colab's local SSD + enabling mixed precision reduces step time to ~0.08 s.

In [ ]:
DRIVE_DATA = "/content/drive/MyDrive/data"   # adjust if your Drive path differs
LOCAL_DATA  = "/content/data"

if not os.path.exists(LOCAL_DATA):
    print("Copying dataset from Drive to local SSD...")
    t0 = time.time()
    os.system(f"cp -r {DRIVE_DATA} {LOCAL_DATA}")
    print(f"Done in {time.time()-t0:.1f}s")
else:
    print("Local data already exists — skipping copy.")

for split in ['train', 'val', 'test']:
    for cls in ['with_mask', 'without_mask']:
        p = os.path.join(LOCAL_DATA, split, cls)
        n = len([f for f in os.listdir(p) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        print(f"  {split}/{cls}: {n} images")

In [ ]:
DATASET_ROOT = LOCAL_DATA
CLASSES      = ["with_mask", "without_mask"]

IMG_SIZE   = 128
BATCH_SIZE = 64
EPOCHS     = 30
SEED       = 42

WEIGHTS_PATH    = "/content/best_mask_cnn.weights.h5"
MODEL_SAVE_PATH = "/content/mask_cnn_final.keras"
TUNER_DIR       = "/content/kt_tuner"
LOG_PATH        = "/content/training_log.csv"
CONFIG_PATH     = "/content/config.json"

# ─── Mixed precision — must be set BEFORE any model is built ─────────────────
tf.keras.mixed_precision.set_global_policy('mixed_float16')
print("Compute policy:", tf.keras.mixed_precision.global_policy().name)
print(f"Batch size : {BATCH_SIZE}")
print(f"Image size : {IMG_SIZE}x{IMG_SIZE}")

---
## 3. Dataset Audit (train / val / test) <a id='3-dataset-audit'></a>

In [ ]:
splits = ['train', 'val', 'test']
counts = {}
for split in splits:
    counts[split] = {}
    for cls in CLASSES:
        p = os.path.join(DATASET_ROOT, split, cls)
        counts[split][cls] = len([
            f for f in os.listdir(p)
            if f.lower().endswith(('.jpg', '.jpeg', '.png'))
        ])

print("=" * 55)
print(f"{'Class':<20} {'Train':>8} {'Val':>8} {'Test':>8} {'Total':>8}")
print("-" * 55)
for cls in CLASSES:
    tr = counts['train'][cls]; va = counts['val'][cls]; te = counts['test'][cls]
    print(f"{cls:<20} {tr:>8} {va:>8} {te:>8} {tr+va+te:>8}")
print("-" * 55)
totals = {sp: sum(counts[sp].values()) for sp in splits}
print(f"{'TOTAL':<20} {totals['train']:>8} {totals['val']:>8} {totals['test']:>8} {sum(totals.values()):>8}")
print("=" * 55)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
colors = ["#2ecc71", "#e74c3c"]
labels = ["With Mask", "Without Mask"]
for ax, split in zip(axes, splits):
    vals = [counts[split][c] for c in CLASSES]
    bars = ax.bar(labels, vals, color=colors, width=0.45, edgecolor='white', linewidth=1.2)
    ax.set_title(split.capitalize(), fontsize=12, fontweight='bold')
    ax.set_ylabel("Images")
    ax.set_ylim(0, max(vals) * 1.25)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(vals)*0.02,
                str(val), ha='center', va='bottom', fontweight='bold')
    ax.spines[['top','right']].set_visible(False)
plt.suptitle("Class Balance — Train / Val / Test", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
def show_sample_grid(root, split, classes, n_per_class=6):
    fig, axes = plt.subplots(len(classes), n_per_class,
                             figsize=(n_per_class * 2.5, len(classes) * 2.5))
    for r, cls in enumerate(classes):
        cls_dir = os.path.join(root, split, cls)
        imgs    = [f for f in os.listdir(cls_dir)
                   if f.lower().endswith((".jpg", ".jpeg", ".png"))]
        samples = random.sample(imgs, min(n_per_class, len(imgs)))
        for c, fname in enumerate(samples):
            img = cv2.imread(os.path.join(cls_dir, fname))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[r, c].imshow(img); axes[r, c].axis('off')
            if c == 0:
                axes[r, c].set_title(cls.replace('_',' ').title(),
                    fontsize=11, fontweight='bold',
                    color='green' if 'with' in cls else 'red')
    plt.suptitle(f"Random Samples — {split} split",
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout(); plt.show()

show_sample_grid(DATASET_ROOT, 'train', CLASSES, n_per_class=6)

In [ ]:
print("Scanning image dimensions (train, first 200 per class)...")
heights, widths, channels_list = [], [], []
for cls in CLASSES:
    cls_dir = os.path.join(DATASET_ROOT, 'train', cls)
    for f in os.listdir(cls_dir)[:200]:
        img = cv2.imread(os.path.join(cls_dir, f))
        if img is not None:
            h, w, ch = img.shape
            heights.append(h); widths.append(w); channels_list.append(ch)

print(f"  Height   — min:{min(heights):4d}  max:{max(heights):4d}  mean:{np.mean(heights):.1f}")
print(f"  Width    — min:{min(widths):4d}  max:{max(widths):4d}  mean:{np.mean(widths):.1f}")
print(f"  Channels (unique): {set(channels_list)}")
print(f"  -> All images resized to {IMG_SIZE}x{IMG_SIZE} px (RGB)")

---
## 4. Preprocessing & tf.data Pipeline <a id='4-pipeline'></a>

### Augmentation applied to train split only

| Transform | Value |
|---|---|
| Random horizontal flip | — |
| Random rotation | ±20° |
| Random zoom | 0–20% |
| Width/height shift | ±15% |
| Brightness | ±30% |
| Contrast | ±20% |
| Rescale | 1/255 |

**Bug fixes applied here vs v1:**
- `decode_image` replaces `decode_jpeg` — handles PNG + JPEG without crashing
- `RandomRotation` instantiated once outside the mapped function — fixes the `tf.Variable inside tf.function` error
- Pipeline order: shuffle → load → cache → augment → batch → prefetch
- All imshow calls cast to float32 — fixes matplotlib dtype rejection of float16

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# ─── FIX: decode_image handles both JPEG and PNG ──────────────────────────────
def load_image(path, label):
    img = tf.io.read_file(path)
    # decode_image works for jpg, jpeg, png, gif, bmp
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

# ─── FIX: RandomRotation instantiated ONCE outside the mapped function ────────
# Instantiating it inside augment() creates a tf.Variable inside a tf.function
# trace, which raises ValueError. This singleton is created at Python level
# before any tf.function tracing happens.
_rot_layer = tf.keras.layers.RandomRotation(
    factor=0.056,           # ±20 degrees (20/360 = 0.056)
    fill_mode='nearest',
    seed=SEED
)

def augment(img, label):
    img = tf.image.random_flip_left_right(img, seed=SEED)
    img = tf.image.random_brightness(img, max_delta=0.3, seed=SEED)
    img = tf.image.random_contrast(img, lower=0.8, upper=1.2, seed=SEED)
    # FIX: tf.stack ensures correct shape for random_crop — avoids scalar issues
    s   = tf.random.uniform((), 0.80, 1.0, seed=SEED)
    h   = tf.cast(tf.cast(IMG_SIZE, tf.float32) * s, tf.int32)
    img = tf.image.random_crop(img, tf.stack([h, h, 3]), seed=SEED)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    # FIX: use pre-instantiated layer — no new Variable created during tracing
    img = _rot_layer(tf.expand_dims(img, 0))[0]
    img = tf.clip_by_value(img, 0.0, 1.0)
    return img, label

def make_dataset(split, augment_data=False, shuffle=True):
    all_paths, all_labels = [], []
    for label_idx, cls in enumerate(CLASSES):
        cls_dir = os.path.join(DATASET_ROOT, split, cls)
        fnames  = [f for f in os.listdir(cls_dir)
                   if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
        for f in fnames:
            all_paths.append(os.path.join(cls_dir, f))
            all_labels.append(float(label_idx))

    ds = tf.data.Dataset.from_tensor_slices((all_paths, all_labels))
    # FIX: shuffle BEFORE cache so path order varies each epoch
    if shuffle:
        ds = ds.shuffle(len(all_paths), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.cache()              # cache decoded+resized images in RAM after epoch 1
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(AUTOTUNE)
    return ds, len(all_paths), all_labels

train_ds, n_train, train_labels = make_dataset('train', augment_data=True,  shuffle=True)
val_ds,   n_val,   val_labels   = make_dataset('val',   augment_data=False, shuffle=False)
test_ds,  n_test,  test_labels  = make_dataset('test',  augment_data=False, shuffle=False)

CLASS_MAP = {i: cls for i, cls in enumerate(CLASSES)}

print("=" * 45)
print(f"  train : {n_train} images  ({n_train // BATCH_SIZE} steps/epoch)")
print(f"  val   : {n_val} images")
print(f"  test  : {n_test} images")
print(f"  classes: {CLASS_MAP}")
print("=" * 45)

In [ ]:
# ─── Visualise augmented samples ─────────────────────────────────────────────
sample_path = os.path.join(DATASET_ROOT, 'train', 'with_mask',
    os.listdir(os.path.join(DATASET_ROOT, 'train', 'with_mask'))[0])
raw_img, _ = load_image(sample_path, 0.0)   # float32 [0,1]

fig, axes = plt.subplots(2, 8, figsize=(18, 5))

# FIX: cast to float32 before imshow — mixed_float16 policy may produce float16
# tensors, and matplotlib imshow only accepts float32/float64 in [0,1]
axes[0, 0].imshow(raw_img.numpy().astype(np.float32))
axes[0, 0].set_title("Original", fontsize=9, color='blue', fontweight='bold')
axes[0, 0].axis('off')

for idx in range(1, 16):
    r, c   = divmod(idx, 8)
    aug, _ = augment(raw_img, 0.0)
    axes[r, c].imshow(aug.numpy().astype(np.float32))   # FIX: cast float16->float32
    axes[r, c].set_title(f"Aug #{idx}", fontsize=8)
    axes[r, c].axis('off')

plt.suptitle("tf.image Augmentation — 15 Versions (float32 [0,1], matching model input)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
labels_np = np.array(train_labels)
cw = sk_class_weight.compute_class_weight(
    class_weight='balanced', classes=np.unique(labels_np), y=labels_np
)
class_weights = {i: float(w) for i, w in enumerate(cw)}
print(f"Class weights: {class_weights}")
print("  (weight > 1 = under-represented class)")

---
## 5. CNN Architecture — Scratch with Residual Blocks <a id='5-cnn-architecture'></a>

**Improvements over v1:**
- `SpatialDropout2D` replaces `Dropout` after conv blocks
- Residual skip connections in Blocks 2 and 3
- `label_smoothing=0.1` on the loss
- Output layer explicitly cast to `float32` (required for `mixed_float16`)

| Block | Layers | Filters |
|---|---|---|
| 1 | Conv→BN→ReLU ×2 → MaxPool | 32 |
| 2 | Conv→BN→ReLU ×2 → MaxPool + **skip** | 64 |
| 3 | Conv→BN→ReLU ×2 → MaxPool + **skip** | 128 |
| 4 (optional) | Conv→BN→ReLU → MaxPool | 256 |
| Head | GlobalAvgPool → Dense → BN → Dropout → Dense(1, sigmoid) | — |

In [ ]:
def build_mask_cnn(
    hp_filters_1=32, hp_filters_2=64, hp_filters_3=128, hp_filters_4=256,
    hp_dense=256, hp_dropout_1=0.4, hp_dropout_2=0.5,
    hp_l2=1e-4, hp_label_smooth=0.1, hp_use_block4=True,
    hp_lr=1e-4
):
    reg = regularizers.l2(hp_l2)
    inp = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="input_image")

    # ── BLOCK 1 ──────────────────────────────────────────────────────────────
    x = layers.Conv2D(hp_filters_1, 3, padding='same', kernel_regularizer=reg, name='B1_conv1')(inp)
    x = layers.BatchNormalization(name='B1_bn1')(x)
    x = layers.Activation('relu', name='B1_relu1')(x)
    x = layers.Conv2D(hp_filters_1, 3, padding='same', kernel_regularizer=reg, name='B1_conv2')(x)
    x = layers.BatchNormalization(name='B1_bn2')(x)
    x = layers.Activation('relu', name='B1_relu2')(x)
    x = layers.MaxPooling2D(2, name='B1_maxpool')(x)

    # ── BLOCK 2 + residual skip ───────────────────────────────────────────────
    skip2 = layers.Conv2D(hp_filters_2, 1, padding='same', name='B2_skip')(x)
    x = layers.Conv2D(hp_filters_2, 3, padding='same', kernel_regularizer=reg, name='B2_conv1')(x)
    x = layers.BatchNormalization(name='B2_bn1')(x)
    x = layers.Activation('relu', name='B2_relu1')(x)
    x = layers.Conv2D(hp_filters_2, 3, padding='same', kernel_regularizer=reg, name='B2_conv2')(x)
    x = layers.BatchNormalization(name='B2_bn2')(x)
    x = layers.Add(name='B2_add')([x, skip2])
    x = layers.Activation('relu', name='B2_relu2')(x)
    x = layers.MaxPooling2D(2, name='B2_maxpool')(x)

    # ── BLOCK 3 + residual skip ───────────────────────────────────────────────
    skip3 = layers.Conv2D(hp_filters_3, 1, padding='same', name='B3_skip')(x)
    x = layers.Conv2D(hp_filters_3, 3, padding='same', kernel_regularizer=reg, name='B3_conv1')(x)
    x = layers.BatchNormalization(name='B3_bn1')(x)
    x = layers.Activation('relu', name='B3_relu1')(x)
    x = layers.Conv2D(hp_filters_3, 3, padding='same', kernel_regularizer=reg, name='B3_conv2')(x)
    x = layers.BatchNormalization(name='B3_bn2')(x)
    x = layers.Add(name='B3_add')([x, skip3])
    x = layers.Activation('relu', name='B3_relu2')(x)
    x = layers.MaxPooling2D(2, name='B3_maxpool')(x)
    x = layers.SpatialDropout2D(hp_dropout_1, name='B3_sdrop')(x)

    # ── BLOCK 4 (optional) ────────────────────────────────────────────────────
    if hp_use_block4:
        x = layers.Conv2D(hp_filters_4, 3, padding='same', kernel_regularizer=reg, name='B4_conv1')(x)
        x = layers.BatchNormalization(name='B4_bn1')(x)
        x = layers.Activation('relu', name='B4_relu1')(x)
        x = layers.MaxPooling2D(2, name='B4_maxpool')(x)

    # ── HEAD ──────────────────────────────────────────────────────────────────
    x   = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x   = layers.Dense(hp_dense, kernel_regularizer=reg, name='dense_head')(x)
    x   = layers.BatchNormalization(name='head_bn')(x)
    x   = layers.Activation('relu', name='head_relu')(x)
    x   = layers.Dropout(hp_dropout_2, name='head_dropout')(x)
    # FIX: explicit float32 cast — required for mixed_float16 numerical stability
    x   = layers.Activation('linear', dtype='float32', name='cast_fp32')(x)
    out = layers.Dense(1, activation='sigmoid', dtype='float32', name='output')(x)

    model = models.Model(inputs=inp, outputs=out, name="MaskCNN_v2")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp_lr),
        loss=keras.losses.BinaryCrossentropy(label_smoothing=hp_label_smooth),
        metrics=[
            'accuracy',
            keras.metrics.Precision(name='precision'),
            keras.metrics.Recall(name='recall'),
            keras.metrics.AUC(name='auc')
        ]
    )
    return model

_tmp = build_mask_cnn()
_tmp.summary()
print(f"\nTotal params: {_tmp.count_params():,}")
del _tmp

---
## 6. Winning Hyperparameters — From Hyperband Search <a id='6-hyperband'></a>

The Hyperband search ran for **1h 24m** across 38+ trials.
Best result: **val_accuracy = 0.9700** (Trial 38).

The search is **complete — do not re-run it**. The winning values are hardcoded
below and passed directly into `build_mask_cnn()`.

| Hyperparameter | Winning Value | Note |
|---|---|---|
| `filters_1` | 16 | Lighter first block |
| `filters_2` | 64 | — |
| `filters_3` | 128 | — |
| `filters_4` | 256 | Irrelevant — Block 4 disabled |
| `dense_units` | 256 | — |
| `dropout_1` (spatial) | 0.25 | Light spatial dropout |
| `dropout_2` (dense) | 0.6 | Strong dense dropout |
| `learning_rate` | 1e-3 | Higher than default — cosine warmup handles it |
| `l2_reg` | 1e-4 | Light L2 |
| `use_block4` | **False** | Removes Block 4 — saves RAM, improves generalisation |
| `label_smoothing` | 0.1 | Prevents overconfidence |

In [ ]:
# ─── Winning hyperparameters — hardcoded from completed Hyperband search ──────
# Best val_accuracy: 0.9700  |  Search time: 1h 24m  |  Trial: 38
# The tuner search is DONE. These values go straight into build_mask_cnn().

best_hp_vals = {
    'filters_1'      : 16,
    'filters_2'      : 64,
    'filters_3'      : 128,
    'filters_4'      : 256,     # unused — use_block4 is False
    'dense_units'    : 256,
    'dropout_1'      : 0.25,
    'dropout_2'      : 0.6,
    'learning_rate'  : 1e-3,
    'l2_reg'         : 1e-4,
    'use_block4'     : False,
    'label_smoothing': 0.1,
}

print("=" * 50)
print("  WINNING HYPERPARAMETERS  (val_accuracy=0.9700)")
print("=" * 50)
for k, v in best_hp_vals.items():
    print(f"  {k:<20}: {v}")
print("=" * 50)
print("\nProceeding directly to final training — no search needed.")

*(Hyperband search cell removed — search is complete. Winner is set above.)*

---
## 7. Final Training with Best Hyperparameters <a id='7-training'></a>

Uses cosine decay with linear warmup instead of ReduceLROnPlateau.
Both `ModelCheckpoint` and `EarlyStopping` monitor `val_loss` (unified — fixes the
v1 monitor mismatch bug).

In [ ]:
import gc
gc.collect()
tf.keras.backend.clear_session()
print("Session cleared — GPU memory freed.")

# ─── Build final model from winning hyperparameters ──────────────────────────
model = build_mask_cnn(
    hp_filters_1    = best_hp_vals['filters_1'],
    hp_filters_2    = best_hp_vals['filters_2'],
    hp_filters_3    = best_hp_vals['filters_3'],
    hp_filters_4    = best_hp_vals['filters_4'],
    hp_dense        = best_hp_vals['dense_units'],
    hp_dropout_1    = best_hp_vals['dropout_1'],
    hp_dropout_2    = best_hp_vals['dropout_2'],
    hp_l2           = best_hp_vals['l2_reg'],
    hp_label_smooth = best_hp_vals['label_smoothing'],
    hp_use_block4   = best_hp_vals['use_block4'],
    hp_lr           = best_hp_vals['learning_rate'],
)
print("Final model built ✓")
model.summary()
print(f"\nTotal params: {model.count_params():,}")

# ─── Cosine decay with linear warmup ─────────────────────────────────────────
STEPS_PER_EPOCH = n_train // BATCH_SIZE
WARMUP_EPOCHS   = 5
WARMUP_STEPS    = WARMUP_EPOCHS * STEPS_PER_EPOCH
TOTAL_STEPS     = EPOCHS * STEPS_PER_EPOCH
BEST_LR         = best_hp_vals['learning_rate']

@tf.keras.utils.register_keras_serializable(package='MaskCNN')
class WarmupCosineSchedule(tf.keras.optimizers.schedules.LearningRateSchedule):
    def __init__(self, initial_lr, warmup_steps, total_steps, alpha=1e-7):
        super().__init__()
        self.initial_lr   = float(initial_lr)
        self.warmup_steps = int(warmup_steps)
        self.total_steps  = int(total_steps)
        self.alpha        = float(alpha)

    def __call__(self, step):
        step      = tf.cast(step, tf.float32)
        warmup    = tf.cast(self.warmup_steps, tf.float32)
        total     = tf.cast(self.total_steps,  tf.float32)
        warmup_lr = self.initial_lr * (step / (warmup + 1e-8))
        progress  = (step - warmup) / tf.maximum(total - warmup, 1.0)
        cosine_lr = self.initial_lr * (
            self.alpha + (1.0 - self.alpha) * 0.5 * (1.0 + tf.cos(np.pi * progress))
        )
        return tf.where(step < warmup, warmup_lr, cosine_lr)

    def get_config(self):
        return {
            'initial_lr'  : self.initial_lr,
            'warmup_steps': self.warmup_steps,
            'total_steps' : self.total_steps,
            'alpha'       : self.alpha
        }

lr_schedule = WarmupCosineSchedule(BEST_LR, WARMUP_STEPS, TOTAL_STEPS)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=lr_schedule),
    loss=keras.losses.BinaryCrossentropy(
        label_smoothing=best_hp_vals['label_smoothing']
    ),
    metrics=[
        'accuracy',
        keras.metrics.Precision(name='precision'),
        keras.metrics.Recall(name='recall'),
        keras.metrics.AUC(name='auc')
    ]
)

cbs = [
    callbacks.ModelCheckpoint(
        filepath=WEIGHTS_PATH,
        monitor='val_loss',
        mode='min',
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.CSVLogger(LOG_PATH)
]

print("\nStarting final training run...")
t0 = time.time()
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=cbs,
    class_weight=class_weights,
    verbose=1
)
print(f"\nTotal training time: {(time.time()-t0)/60:.1f} minutes")

---
## 8. Training Curves & Metrics <a id='8-metrics'></a>

In [ ]:
def plot_history(history):
    h      = history.history
    epochs = range(1, len(h['loss']) + 1)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("Training History", fontsize=16, fontweight='bold')
    metrics = [
        ('loss',      'Loss',      '#e74c3c'),
        ('accuracy',  'Accuracy',  '#2ecc71'),
        ('precision', 'Precision', '#3498db'),
        ('recall',    'Recall',    '#9b59b6'),
        ('auc',       'AUC',       '#f39c12'),
    ]
    for ax, (key, title, color) in zip(axes.flat[:5], metrics):
        ax.plot(epochs, h[key],          color=color, lw=2, label='Train')
        ax.plot(epochs, h[f'val_{key}'], color=color, lw=2,
                linestyle='--', alpha=0.7, label='Val')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Epoch'); ax.legend()
        ax.spines[['top','right']].set_visible(False)
    axes[1,2].axis('off')
    plt.tight_layout(); plt.show()

plot_history(history)

h = history.history
best_ep = int(np.argmax(h['val_accuracy'])) + 1
print("=" * 50)
print(f"  BEST EPOCH: {best_ep}")
print("=" * 50)
for metric in ['loss','accuracy','precision','recall','auc']:
    print(f"  {metric:<12}  train={h[metric][best_ep-1]:.4f}   val={h[f'val_{metric}'][best_ep-1]:.4f}")
print("=" * 50)

---
## 9. Validation Evaluation <a id='9-val-evaluation'></a>

Evaluation on the validation set used during training.
Small optimistic bias from checkpoint selection — see Section 10 for unbiased numbers.

In [ ]:
model.load_weights(WEIGHTS_PATH)

y_prob_val = model.predict(val_ds, verbose=1).ravel()
y_pred_val = (y_prob_val >= 0.5).astype(int)
y_true_val = np.array(val_labels)
class_names = [CLASS_MAP[0].replace('_',' ').title(),
               CLASS_MAP[1].replace('_',' ').title()]

cm_val = confusion_matrix(y_true_val, y_pred_val)
tn, fp, fn, tp = cm_val.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay(cm_val, display_labels=class_names).plot(
    ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title("Confusion Matrix — Counts (Val)", fontweight='bold')

cm_norm = cm_val.astype(float) / cm_val.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(np.round(cm_norm,3), display_labels=class_names).plot(
    ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title("Confusion Matrix — Normalised (Val)", fontweight='bold')
plt.tight_layout(); plt.show()

print(classification_report(y_true_val, y_pred_val, target_names=class_names))

fpr_val, tpr_val, _ = roc_curve(y_true_val, y_prob_val)
roc_auc_val = auc(fpr_val, tpr_val)

plt.figure(figsize=(7,6))
plt.plot(fpr_val, tpr_val, color='#2980b9', lw=2, label=f"AUC = {roc_auc_val:.4f}")
plt.plot([0,1],[0,1],'k--',lw=1)
plt.fill_between(fpr_val, tpr_val, alpha=0.1, color='#2980b9')
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Validation Set", fontweight='bold')
plt.legend(); plt.gca().spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
thresholds_check = np.arange(0.1, 1.0, 0.05)
precisions, recalls, f1s, accs = [], [], [], []

for thr in thresholds_check:
    preds = (y_prob_val >= thr).astype(int)
    precisions.append(precision_score(y_true_val, preds, zero_division=0))
    recalls.append(recall_score(y_true_val, preds, zero_division=0))
    f1s.append(f1_score(y_true_val, preds, zero_division=0))
    accs.append(accuracy_score(y_true_val, preds))

best_thr_idx   = int(np.argmax(f1s))
BEST_THRESHOLD = float(thresholds_check[best_thr_idx])

plt.figure(figsize=(10,5))
plt.plot(thresholds_check, precisions, label='Precision', color='#3498db', lw=2)
plt.plot(thresholds_check, recalls,    label='Recall',    color='#e74c3c', lw=2)
plt.plot(thresholds_check, f1s,        label='F1-Score',  color='#2ecc71', lw=2)
plt.plot(thresholds_check, accs,       label='Accuracy',  color='#9b59b6', lw=2)
plt.axvline(0.5,            color='gray',  linestyle='--', alpha=0.5, label='Default (0.5)')
plt.axvline(BEST_THRESHOLD, color='black', linestyle='-',  alpha=0.8,
            label=f'Optimal ({BEST_THRESHOLD:.2f})')
plt.xlabel("Decision Threshold"); plt.ylabel("Metric Value")
plt.title("Threshold Sensitivity Analysis", fontweight='bold')
plt.legend(); plt.tight_layout(); plt.show()

print(f"Optimal threshold (max F1): {BEST_THRESHOLD:.2f}")
print(f"  Precision : {precisions[best_thr_idx]:.4f}")
print(f"  Recall    : {recalls[best_thr_idx]:.4f}")
print(f"  F1-Score  : {f1s[best_thr_idx]:.4f}")
print(f"  Accuracy  : {accs[best_thr_idx]:.4f}")

---
## 10. Test Set Evaluation — Unbiased Final Metrics <a id='10-test-evaluation'></a>

The test set was never seen during training or monitoring.
The optimal threshold from Section 9 is applied here.

In [ ]:
y_prob_test = model.predict(test_ds, verbose=1).ravel()
y_pred_test = (y_prob_test >= BEST_THRESHOLD).astype(int)
y_true_test = np.array(test_labels)

cm_test = confusion_matrix(y_true_test, y_pred_test)
tn_t, fp_t, fn_t, tp_t = cm_test.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay(cm_test, display_labels=class_names).plot(
    ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title("Confusion Matrix — Counts (TEST)", fontweight='bold')

cm_test_norm = cm_test.astype(float) / cm_test.sum(axis=1, keepdims=True)
ConfusionMatrixDisplay(np.round(cm_test_norm,3), display_labels=class_names).plot(
    ax=axes[1], cmap='Greens', colorbar=False)
axes[1].set_title("Confusion Matrix — Normalised (TEST)", fontweight='bold')
plt.tight_layout(); plt.show()

print("=" * 55)
print("  UNBIASED TEST SET METRICS")
print("=" * 55)
print(classification_report(y_true_test, y_pred_test, target_names=class_names))

fpr_test, tpr_test, _ = roc_curve(y_true_test, y_prob_test)
roc_auc_test = auc(fpr_test, tpr_test)

plt.figure(figsize=(7,6))
plt.plot(fpr_test, tpr_test, color='#e74c3c', lw=2, label=f"AUC = {roc_auc_test:.4f}")
plt.plot([0,1],[0,1],'k--',lw=1)
plt.fill_between(fpr_test, tpr_test, alpha=0.1, color='#e74c3c')
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve — Test Set (Unbiased)", fontweight='bold')
plt.legend(); plt.gca().spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

test_metrics = {
    "accuracy":        float(accuracy_score(y_true_test, y_pred_test)),
    "precision":       float(precision_score(y_true_test, y_pred_test, zero_division=0)),
    "recall":          float(recall_score(y_true_test, y_pred_test, zero_division=0)),
    "f1_score":        float(f1_score(y_true_test, y_pred_test, zero_division=0)),
    "roc_auc":         float(roc_auc_test),
    "threshold_used":  float(BEST_THRESHOLD),
    "tp": int(tp_t), "tn": int(tn_t), "fp": int(fp_t), "fn": int(fn_t)
}
print("\nTest metrics stored ✓")

---
## 11. MaxPool Feature Map Visualisation <a id='11-feature-maps'></a>

In [ ]:
maxpool_names = [l.name for l in model.layers if 'maxpool' in l.name]
print("MaxPool layers:", maxpool_names)

feature_models = {
    name: models.Model(inputs=model.input, outputs=model.get_layer(name).output)
    for name in maxpool_names
}

sample_path = os.path.join(DATASET_ROOT, 'test', 'with_mask',
    os.listdir(os.path.join(DATASET_ROOT, 'test', 'with_mask'))[0])
raw_np  = cv2.imread(sample_path)
raw_rgb = cv2.cvtColor(raw_np, cv2.COLOR_BGR2RGB)
resized = cv2.resize(raw_rgb, (IMG_SIZE, IMG_SIZE))
sample_t = np.expand_dims(resized.astype('float32') / 255.0, 0)

prob  = float(model(sample_t, training=False).numpy()[0][0])
label = CLASS_MAP[int(prob >= BEST_THRESHOLD)].replace('_',' ').title()

plt.figure(figsize=(3,3))
plt.imshow(resized)   # uint8 [0-255] — imshow handles this correctly
plt.title(f"Sample -> {label}\n(prob={prob:.3f})", fontweight='bold',
          color='green' if 'With' in label else 'red')
plt.axis('off'); plt.tight_layout(); plt.show()

# ─── Feature map grid ─────────────────────────────────────────────────────────
fig  = plt.figure(figsize=(20, len(feature_models)*3.5))
fig.suptitle("MaxPool Feature Maps", fontsize=15, fontweight='bold', y=1.01)
outer = gridspec.GridSpec(len(feature_models), 1, hspace=0.6)

for row_idx, (lname, fm_model) in enumerate(feature_models.items()):
    # FIX: cast float16 -> float32 before passing to matplotlib
    fmaps  = fm_model(sample_t, training=False).numpy()[0].astype(np.float32)
    n_cols = min(16, fmaps.shape[-1])
    inner  = gridspec.GridSpecFromSubplotSpec(1, n_cols+1,
                 subplot_spec=outer[row_idx], wspace=0.05)
    ax0 = fig.add_subplot(inner[0])
    ax0.text(0.5, 0.5, f"{lname}\n{fmaps.shape[0]}x{fmaps.shape[1]}\n{fmaps.shape[2]}ch",
             ha='center', va='center', fontsize=9, fontweight='bold',
             color='white', transform=ax0.transAxes)
    ax0.set_facecolor('#2c3e50'); ax0.axis('off')
    for c in range(n_cols):
        ax = fig.add_subplot(inner[c+1])
        ax.imshow(fmaps[:,:,c], cmap='viridis', interpolation='nearest')
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"ch{c}", fontsize=6, pad=1)
plt.show()

# ─── Mean / Max heatmaps ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, len(feature_models), figsize=(len(feature_models)*4, 8))
for idx, (lname, fm_model) in enumerate(feature_models.items()):
    # FIX: cast float16 -> float32
    fmaps = fm_model(sample_t, training=False).numpy()[0].astype(np.float32)
    axes[0,idx].imshow(np.mean(fmaps,axis=-1), cmap='hot', interpolation='bilinear')
    axes[0,idx].set_title(f"{lname}\nMean activation", fontsize=9, fontweight='bold')
    axes[0,idx].axis('off')
    axes[1,idx].imshow(np.max(fmaps,axis=-1),  cmap='hot', interpolation='bilinear')
    axes[1,idx].set_title(f"{lname}\nMax activation",  fontsize=9, fontweight='bold')
    axes[1,idx].axis('off')
plt.suptitle("Mean & Max Activation per MaxPool Layer", fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
## 12. Grad-CAM Explainability <a id='12-gradcam'></a>

Shows which pixels drove the classification. For a reliable mask detector the
heatmap should highlight the mouth/nose region.

In [ ]:
def grad_cam(model, img_tensor, target_layer_name, class_idx=0):
    """
    Compute Grad-CAM heatmap.
    img_tensor : (1, H, W, 3) float32 numpy array in [0,1]
    Returns    : (H, W) float32 numpy array in [0,1]
    """
    grad_model = tf.keras.Model(
        inputs=model.input,
        outputs=[model.get_layer(target_layer_name).output, model.output]
    )
    img_t = tf.cast(img_tensor, tf.float32)
    with tf.GradientTape() as tape:
        # FIX: explicitly watch conv_out — it is a Tensor, not a Variable,
        # so GradientTape does not watch it automatically.
        # Without this, tape.gradient returns None and [0] raises AttributeError.
        tape.watch(img_t)
        conv_out, preds = grad_model(img_t, training=False)
        tape.watch(conv_out)
        loss = preds[:, class_idx]

    grads   = tape.gradient(loss, conv_out)      # (1, H, W, C)
    grads   = grads[0]                           # (H, W, C)
    weights = tf.reduce_mean(grads, axis=(0, 1)) # (C,)
    cam     = tf.reduce_sum(weights * conv_out[0], axis=-1)   # (H, W)
    cam     = tf.maximum(cam, 0.0)
    cam     = cam / (tf.math.reduce_max(cam) + 1e-8)
    return cam.numpy().astype(np.float32)


def show_gradcam_grid(model, dataset_root, classes, n_per_class=3):
    last_conv = [l.name for l in model.layers if isinstance(l, layers.Conv2D)][-1]
    print(f"Grad-CAM target layer: {last_conv}")

    n_rows = len(classes) * n_per_class
    fig, axes = plt.subplots(n_rows, 3, figsize=(10, n_rows * 3))
    fig.suptitle(f"Grad-CAM — layer: {last_conv}", fontsize=13, fontweight='bold')

    row = 0
    for cls in classes:
        cls_dir = os.path.join(dataset_root, 'test', cls)
        fnames  = [f for f in os.listdir(cls_dir)
                   if f.lower().endswith(('.jpg','.jpeg','.png'))][:n_per_class]
        for fname in fnames:
            img_bgr = cv2.imread(os.path.join(cls_dir, fname))
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            img_res = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE))
            tensor  = np.expand_dims(img_res.astype('float32') / 255.0, 0)

            prob  = float(model(tensor, training=False).numpy()[0][0])
            pred  = CLASS_MAP[int(prob >= BEST_THRESHOLD)].replace('_',' ').title()
            truth = cls.replace('_',' ').title()

            cam     = grad_cam(model, tensor, last_conv, class_idx=0)
            cam_big = cv2.resize(cam, (IMG_SIZE, IMG_SIZE))
            heatmap = cv2.applyColorMap((cam_big * 255).astype(np.uint8), cv2.COLORMAP_JET)
            heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
            # both img_res (uint8) and heatmap (uint8) — addWeighted works correctly
            overlay = cv2.addWeighted(img_res, 0.55, heatmap, 0.45, 0)

            correct = pred.lower().replace(' ','_') == cls
            color   = 'green' if correct else 'red'

            axes[row,0].imshow(img_res);  axes[row,0].axis('off')
            axes[row,0].set_title(f"Input\nTrue: {truth}", fontsize=8)
            axes[row,1].imshow(cam_big, cmap='hot'); axes[row,1].axis('off')
            axes[row,1].set_title("Grad-CAM", fontsize=8)
            axes[row,2].imshow(overlay); axes[row,2].axis('off')
            axes[row,2].set_title(f"Overlay\nPred: {pred} ({prob:.2f})",
                                  fontsize=8, color=color)
            row += 1

    plt.tight_layout(); plt.show()


show_gradcam_grid(model, DATASET_ROOT, CLASSES, n_per_class=3)

---
## 13. Save Model & config.json <a id='13-save'></a>

In [ ]:
model.save(MODEL_SAVE_PATH)
print(f"Model saved -> {MODEL_SAVE_PATH}")

best_ep_idx = int(np.argmax(history.history['val_accuracy']))

config = {
    "model_name" : "MaskCNN_v2",
    "version"    : "2.0.0",
    "created_at" : time.strftime("%Y-%m-%d %H:%M:%S"),

    "architecture": {
        "input_shape"          : [IMG_SIZE, IMG_SIZE, 3],
        "conv_blocks"          : 3,   # Block 4 disabled by use_block4=False
        "residual_skips"       : "Blocks 2 and 3",
        "spatial_dropout"      : True,
        "global_avg_pool"      : True,
        "batch_normalization"  : True,
        "total_params"         : int(model.count_params()),
        "winning_hyperparameters": {
            k: (bool(v) if isinstance(v, bool) else v)
            for k, v in best_hp_vals.items()
        }
    },

    "training": {
        "optimizer"              : "Adam + WarmupCosineSchedule",
        "loss"                   : "BinaryCrossentropy(label_smoothing=0.1)",
        "epochs_configured"      : EPOCHS,
        "epochs_run"             : len(history.history['loss']),
        "best_epoch"             : best_ep_idx + 1,
        "batch_size"             : BATCH_SIZE,
        "mixed_precision"        : "float16",
        "class_weights"          : class_weights,
        "early_stopping_patience": 8,
        "warmup_epochs"          : WARMUP_EPOCHS,
        "hyperband_search"       : {
            "completed"          : True,
            "best_val_accuracy"  : 0.9700,
            "trials_run"         : 38,
            "search_time_minutes": 84
        }
    },

    "augmentation": {
        "pipeline"  : "tf.data + tf.image (GPU-accelerated)",
        "applied_to": "train split only",
        "transforms": ["random_flip_left_right", "random_brightness",
                       "random_contrast", "random_zoom_crop", "random_rotation"]
    },

    "dataset": {
        "structure"   : "pre-split train/val/test",
        "train_images": n_train,
        "val_images"  : n_val,
        "test_images" : n_test,
        "classes"     : CLASSES
    },

    "val_metrics": {
        "accuracy"  : float(accuracy_score(y_true_val, y_pred_val)),
        "precision" : float(precision_score(y_true_val, y_pred_val, zero_division=0)),
        "recall"    : float(recall_score(y_true_val, y_pred_val, zero_division=0)),
        "f1_score"  : float(f1_score(y_true_val, y_pred_val, zero_division=0)),
        "roc_auc"   : float(roc_auc_val),
        "note"      : "Slight optimistic bias from checkpoint selection"
    },

    "test_metrics": test_metrics,

    "inference": {
        "classes"           : CLASSES,
        "class_map"         : {str(k): v for k, v in CLASS_MAP.items()},
        "decision_threshold": float(BEST_THRESHOLD),
        "input_size"        : [IMG_SIZE, IMG_SIZE]
    },

    "files": {
        "model"        : MODEL_SAVE_PATH,
        "best_weights" : WEIGHTS_PATH,
        "training_log" : LOG_PATH
    }
}

with open(CONFIG_PATH, 'w') as f:
    json.dump(config, f, indent=2)
print(f"config.json saved -> {CONFIG_PATH}")
print(json.dumps(config, indent=2))

from google.colab import files
files.download(MODEL_SAVE_PATH)
files.download(CONFIG_PATH)

---
## 14. Real-Time Detection — Webcam / Video <a id='14-realtime'></a>

Uses OpenCV DNN SSD face detector (2017) instead of Haar Cascade (2001).
Handles faces at angles up to ~70° and partially occluded faces.

In [ ]:
# ─── Download OpenCV DNN face detector weights ───────────────────────────────
# FIX: opencv_3rdparty raw URL has moved. Using the opencv/opencv master branch
# deploy.prototxt and a reliable caffemodel mirror.

PROTO_PATH = "/content/deploy.prototxt"
MODEL_PATH = "/content/res10_300x300_ssd.caffemodel"

import urllib.request

PROTO_URL = ("https://raw.githubusercontent.com/opencv/opencv/master/"
             "samples/dnn/face_detector/deploy.prototxt")

# Caffemodel from the official opencv releases page (stable)
MODEL_URL = ("https://github.com/opencv/opencv_3rdparty/raw/"
             "dnn_samples_face_detector_20170830/"
             "res10_300x300_ssd_iter_140000.caffemodel")

for url, path, name in [
    (PROTO_URL, PROTO_PATH, "deploy.prototxt"),
    (MODEL_URL, MODEL_PATH, "res10_300x300_ssd.caffemodel"),
]:
    if not os.path.exists(path):
        print(f"Downloading {name}...")
        try:
            urllib.request.urlretrieve(url, path)
            print(f"  OK -> {path}  ({os.path.getsize(path):,} bytes)")
        except Exception as e:
            print(f"  Download failed: {e}")
            print("  Falling back to gdown...")
            # gdrive fallback IDs for the caffemodel
            if "caffemodel" in name:
                os.system("gdown --id 1DkuBHlQJblIIbRHCk_dIWwW0pTqvQqOq "
                          f"-O {path}")
    else:
        print(f"Already exists: {path}  ({os.path.getsize(path):,} bytes)")

face_net = cv2.dnn.readNetFromCaffe(PROTO_PATH, MODEL_PATH)
print("DNN face detector loaded ✓")

In [ ]:
with open(CONFIG_PATH) as f:
    cfg = json.load(f)

THRESHOLD     = cfg['inference']['decision_threshold']
INFER_SIZE    = tuple(cfg['inference']['input_size'])
CLASS_MAP_INF = {int(k): v for k, v in cfg['inference']['class_map'].items()}

loaded_model = keras.models.load_model(
    MODEL_SAVE_PATH,
    custom_objects={'WarmupCosineSchedule': WarmupCosineSchedule}
)
print(f"Model loaded ✓  (threshold={THRESHOLD})")

In [ ]:
def detect_faces_dnn(frame, conf_threshold=0.5):
    h, w = frame.shape[:2]
    blob = cv2.dnn.blobFromImage(frame, 1.0, (300,300), (104,177,123))
    face_net.setInput(blob)
    dets = face_net.forward()
    faces = []
    for i in range(dets.shape[2]):
        if dets[0,0,i,2] > conf_threshold:
            box = dets[0,0,i,3:7] * np.array([w,h,w,h])
            x1,y1,x2,y2 = box.astype(int)
            x1,y1 = max(0,x1), max(0,y1)
            x2,y2 = min(w,x2), min(h,y2)
            if x2 > x1 and y2 > y1:
                faces.append((x1, y1, x2-x1, y2-y1))
    return faces


def preprocess_face(face_bgr):
    rgb  = cv2.cvtColor(face_bgr, cv2.COLOR_BGR2RGB)
    rsz  = cv2.resize(rgb, INFER_SIZE)
    norm = rsz.astype('float32') / 255.0
    return np.expand_dims(norm, axis=0)


def annotate_frame(frame, faces, predictions):
    for (x,y,w,h), (prob, label) in zip(faces, predictions):
        wearing = 'with_mask' in label
        color   = (0, 200, 0) if wearing else (0, 0, 220)
        conf    = prob if wearing else 1 - prob
        text    = f"{label.replace('_',' ').title()} {conf*100:.1f}%"
        if not wearing:
            overlay = frame.copy()
            cv2.rectangle(overlay, (x,y), (x+w,y+h), (0,0,180), -1)
            frame = cv2.addWeighted(overlay, 0.25, frame, 0.75, 0)
        cv2.rectangle(frame, (x,y), (x+w,y+h), color, 2)
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 2)
        cv2.rectangle(frame, (x, y-th-8), (x+tw+4, y), color, -1)
        cv2.putText(frame, text, (x+2, y-4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 2)
    return frame


def run_detection(source=0, save_output=None, max_frames=None):
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print(f"Could not open source: {source}"); return

    fps    = cap.get(cv2.CAP_PROP_FPS) or 30
    W      = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H      = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    writer = None
    if save_output:
        writer = cv2.VideoWriter(
            save_output, cv2.VideoWriter_fourcc(*'mp4v'), fps, (W, H))
        print(f"Saving to {save_output}")

    frame_count = total_faces = mask_count = 0
    t0 = time.time()
    print("Press Q to quit")

    while True:
        ret, frame = cap.read()
        if not ret: break
        frame_count += 1
        if max_frames and frame_count > max_frames: break

        faces = detect_faces_dnn(frame)
        predictions = []
        if faces:
            batch = np.concatenate(
                [preprocess_face(frame[y:y+h, x:x+w]) for (x,y,w,h) in faces], axis=0)
            probs = loaded_model(batch, training=False).numpy().ravel()
            for prob in probs:
                label = CLASS_MAP_INF[int(prob >= THRESHOLD)]
                predictions.append((float(prob), label))
                total_faces += 1
                if 'with_mask' in label: mask_count += 1

        annotate_frame(frame, faces, predictions)
        cur_fps = frame_count / (time.time() - t0 + 1e-6)
        cv2.putText(frame, f"FPS: {cur_fps:.1f}", (10, H-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 2)
        if writer: writer.write(frame)
        cv2.imshow('Mask Detection', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

    cap.release()
    if writer: writer.release()
    cv2.destroyAllWindows()
    print(f"Frames: {frame_count} | Faces: {total_faces} | "
          f"Masked: {mask_count} | Unmasked: {total_faces-mask_count}")
    if total_faces > 0:
        print(f"Mask compliance: {mask_count/total_faces*100:.1f}%")

print("run_detection() defined ✓")
print("Usage: run_detection(source=0)                    # webcam")
print("       run_detection(source='video.mp4',")
print("                     save_output='out.mp4')       # video file")

---
## Bug Fixes Applied in this Version

| Cell | Bug | Fix |
|---|---|---|
| 13 | `RandomRotation` creates `tf.Variable` inside `tf.function` trace | Instantiate `_rot_layer` once outside `augment()` |
| 13 | `decode_jpeg` crashes on PNG files | Replaced with `decode_image(expand_animations=False)` |
| 13 | `random_crop` size argument could be malformed | Use `tf.stack([h, h, 3])` |
| 14 | `imshow` rejects `float16` arrays | Cast `.numpy().astype(np.float32)` before all `imshow` calls |
| 19 | `optimizer.learning_rate.assign()` raises `AttributeError` | Pass `hp_lr` into `build_mask_cnn` and compile there |
| 22 | `WarmupCosineSchedule.get_config()` calls `.numpy()` during save | Store all values as plain Python int/float in `__init__` |
| 22 | Custom schedule not serialisable | Add `@register_keras_serializable` decorator |
| 31 | Feature map arrays are `float16` — matplotlib rejects them | Cast `.astype(np.float32)` after every `.numpy()` call |
| 33 | `tape.gradient` returns `None` for non-Variable tensors | Add `tape.watch(conv_out)` inside the `GradientTape` block |
| 37 | DNN model URL returns 404 | Use `github.com/opencv/opencv_3rdparty/raw/...` with gdown fallback |
| 38 | `load_model` can't deserialise custom schedule | Pass `custom_objects={'WarmupCosineSchedule': WarmupCosineSchedule}` |

---
*End of notebook v2.1*
